<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/Control%20Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Test


In [1]:
# @title Env

!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

# %%
from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

# %%
# Optional: install control (not used in this notebook but kept for compatibility)
!pip install -q control

# ## 2. Load Model (IDF + Weather)


import types, datetime, requests, io, os, gc
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import traceback
from eplus.core import EPlusUtil

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 578.3/578.3 kB 12.5 MB/s eta 0:00:00


In [8]:
# @title Setup
OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/5ZoneCAV_MaxTemp.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)

sim.ensure_output_sqlite()
sim.prepare_run_with_co2(
    outdoor_co2_ppm=420.0,
    wipe_outputs=True,
    activate=True,
    reset=True
)

required_vars = [
    # Environment
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "Environment", "freq": "Timestep"},
    {"name": "Site Outdoor Air Humidity Ratio", "key": "Environment", "freq": "Timestep"},

    # AHU Nodes & Components
    {"name": "System Node Temperature", "key": "*", "freq": "Timestep"},
    {"name": "System Node Mass Flow Rate", "key": "*", "freq": "Timestep"},
    {"name": "Cooling Coil Total Cooling Rate", "key": "*", "freq": "Timestep"},
    {"name": "Heating Coil Heating Rate", "key": "*", "freq": "Timestep"},
    {"name": "Fan Electricity Rate", "key": "*", "freq": "Timestep"},

    # Zone States & Parameters (Adding your Occupancy & Equipment requests)
    {"name": "Zone Air Temperature", "key": "*", "freq": "Timestep"},
    {"name": "Zone Mean Radiant Temperature", "key": "*", "freq": "Timestep"}, # Proxy for Tm
    {"name": "Zone Mean Air Humidity Ratio", "key": "*", "freq": "Timestep"},
    {"name": "Zone Air CO2 Concentration", "key": "*", "freq": "Timestep"},
    {"name": "Zone People Occupant Count", "key": "*", "freq": "Timestep"},
    {"name": "Zone Electric Equipment Total Heating Rate", "key": "*", "freq": "Timestep"}
]

print("Patching IDF with Timestep Output Variables...")
sim.ensure_output_variables(required_vars)

print("Executing Dry Run...")
sim.run_dry_run(include_ems_edd=False, reset=True, design_day=True)
print("Dry Run Complete!")



Initialized StateMixin
Initialized EnergyPlus State.
Initialized IDFMixin
Initialized LoggingMixin
Initialized SimulationMixin
Initialized UtilsMixin
Initialized HandlersMixin
Initialized SQLMixin
Initialized ControlMixin
Initialized OccupancyMixin
Initialized ZoneObserverMixin
EnergyPlus state has been reset.
Deleted output directory: /simulation/eplus_out
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/5ZoneCAV_MaxTemp.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.
EnergyPlus state has been reset.
Patching IDF with Timestep Output Variables...
EnergyPlus state has been reset.
Executing Dry Run...
EnergyPlus state has been reset.
Dry Run Complete!


In [9]:
# @title Occupancy and CO2
def preload_occupancy_csv(sim_obj, url):
    """
    Downloads the CSV, anchors time to midnight, and forces a perfect 24-hour loop.
    """
    print(f"Downloading CSV from: {url}...")
    try:
        resp = requests.get(url)
        resp.raise_for_status()

        df = pd.read_csv(io.StringIO(resp.text))
        if 'timestamp' not in df.columns:
            raise ValueError("CSV must contain a 'timestamp' column.")

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')

        # 1. Anchor to MIDNIGHT of the first day to prevent timestep offset
        midnight_start = df['timestamp'].iloc[0].normalize()
        df['rel_seconds'] = (df['timestamp'] - midnight_start).dt.total_seconds()

        # 2. Force exactly 24 hours for daily looping to prevent modulo drift
        sim_obj._occ_duration_sec = 86400.0

        # 3. Clean up dataframe
        df = df.set_index('rel_seconds')
        sim_obj._preloaded_occ_df = df.drop(columns=['timestamp'])

        zones = list(sim_obj._preloaded_occ_df.columns)
        print(f"Success! Preloaded {len(df)} rows. Loop locked to 24.00 hours.")
        print(f"Detected Source Columns: {zones}")

    except Exception as e:
        print(f"Failed to preload CSV: {e}")
# Load CSV
csv_url = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/Occupancy_Dataset.csv"
preload_occupancy_csv(sim, csv_url)

def people_injector(self, state):
    """
    Lightning-fast runtime handler. Maps actuators on the first tick.
    Reads a single baseline zone from the CSV and uses multipliers,
    ceilings, and clipping to populate other zones synthetically.
    """
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. One-Time Setup: Map Actuators & Define Extrapolation Rules ---
    if not hasattr(self, '_fast_injector_ready'):
        # Ensure the data was preloaded via Step 1
        if not hasattr(self, '_preloaded_occ_df'):
            print("[Injector] ERROR: Data not preloaded. Run preload_occupancy_csv() first.")
            self._fast_injector_ready = False
            return

        # =========================================================
        # CONFIGURATION DICTIONARY: TWEAK YOUR MULTIPLIERS HERE
        # source: The CSV column to read the baseline value from
        # mult: The multiplier applied to the source value
        # min/max: The clipping bounds to enforce physical limits
        # =========================================================
        self._zone_occ_rules = {
            "SPACE1-1": {"source": "SPACE1-1", "mult": 1.0,  "min": 0, "max": 5}, # Baseline
            "SPACE2-1": {"source": "SPACE1-1", "mult": 1.5,  "min": 0, "max": 4},
            "SPACE3-1": {"source": "SPACE1-1", "mult": 0.4,  "min": 0, "max": 1},
            "SPACE4-1": {"source": "SPACE1-1", "mult": 1.2,  "min": 0, "max": 3},
            "SPACE5-1": {"source": "SPACE1-1", "mult": 2.0,  "min": 0, "max": 6},
        }

        self._people_handles = {}
        target_zones = list(self._zone_occ_rules.keys())

        try:
            ep_people_names = self.exchange.get_object_names(state, "People") or []
        except Exception:
            ep_people_names = []

        # Map to EnergyPlus Actuators based on the target zones, NOT just CSV columns
        mapped_count = 0
        for z in target_zones:
            matched_people = [p for p in ep_people_names if z.replace(" ", "").lower() in p.replace(" ", "").lower()]
            handles = []
            for p in matched_people:
                h = self.exchange.get_actuator_handle(state, "People", "Number of People", p)
                if h != -1:
                    handles.append(h)
                    mapped_count += 1
            if handles:
                self._people_handles[z] = handles

        print(f"\n[Injector] Mapped {mapped_count} actuators across {len(self._people_handles)} zones (Extrapolation Active).")

        # Mark exact start time of the simulation
        day = self.exchange.day_of_year(state)
        time_hr = self.exchange.current_time(state)
        self._sim_start_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

        self._fast_injector_ready = True

    # --- Runtime Safety Check ---
    if not self._fast_injector_ready or getattr(self, '_occ_duration_sec', 0) == 0:
        return

    # --- 2. Calculate Elapsed Time & Loop ---
    day = self.exchange.day_of_year(state)
    time_hr = self.exchange.current_time(state)
    current_date = datetime.datetime(2002, 1, 1) + datetime.timedelta(days=day - 1, seconds=(int(time_hr * 3600)))

    elapsed_seconds = (current_date - self._sim_start_date).total_seconds()
    loop_sec = elapsed_seconds % self._occ_duration_sec

    # --- 3. Fast Data Lookup (Forward Fill) ---
    df = self._preloaded_occ_df
    valid_indices = df.index[df.index <= loop_sec]
    target_idx = df.index[0] if len(valid_indices) == 0 else valid_indices[-1]
    row = df.loc[target_idx]

    # --- 4. Extrapolate and Inject Values ---
    for z, handles in self._people_handles.items():
        rule = self._zone_occ_rules.get(z)
        if not rule:
            continue

        src_col = rule["source"]
        if src_col in row:
            base_val = float(row[src_col])

            # Apply math: Base * Multiplier -> Round Up -> Clip
            if base_val == 0:
                val = 0.0 # Bypasses math to strictly enforce zero at night
            else:
                calculated = np.ceil(base_val * rule["mult"])
                val = float(np.clip(calculated, rule["min"], rule["max"]))

            # Divide evenly if there are multiple People objects in the same room
            per_actuator = val / len(handles)
            for h in handles:
                self.exchange.set_actuator_value(state, h, per_actuator)

# --- Registration ---
sim.people_injector = types.MethodType(people_injector, sim)

sim.register_handlers("begin", [
    {"method_name": "people_injector"},
])

print(f"Handlers on 'begin' hook: {sim.list_handlers("begin")}")

Success! Preloaded 10129 rows. Loop locked to 24.00 hours.
Detected Source Columns: ['SPACE1-1']
Handlers on 'begin' hook: ['people_injector']


In [10]:
# @title Simulation Logger
def simulation_logger(self, state):
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    # --- 1. One-Time Setup: Resolve and Cache Handles ---
    if not hasattr(self, '_fast_logger_ready'):
        self.sim_log_data = []
        self._var_handles = {}

        def get_vh(type_name, key):
            h = self.exchange.get_variable_handle(state, type_name, key)
            if h <= 0:
                print(f"[Logger] Warning: Could not map variable '{type_name}' for '{key}'")
            return h

        print("\n[Logger] Mapping EnergyPlus handles for 5ZoneCAV_MaxTemp.idf...")

        # Environment
        self._var_handles['Env_Temp_Out'] = get_vh("Site Outdoor Air Drybulb Temperature", "Environment")
        self._var_handles['Env_Humid_Out'] = get_vh("Site Outdoor Air Humidity Ratio", "Environment")

        # AHU
        self._var_handles['AHU_Mix_Temp'] = get_vh("System Node Temperature", "Mixed Air Node 1")
        self._var_handles['AHU_Cool_Out_Temp'] = get_vh("System Node Temperature", "Main Cooling Coil 1 Outlet Node")
        self._var_handles['AHU_Heat_Out_Temp'] = get_vh("System Node Temperature", "Main Heating Coil 1 Outlet Node")
        self._var_handles['AHU_Supply_Temp'] = get_vh("System Node Temperature", "VAV Sys 1 Outlet Node")

        self._var_handles['AHU_Cooling_Rate'] = get_vh("Cooling Coil Total Cooling Rate", "Main Cooling Coil 1")
        self._var_handles['AHU_Heating_Rate'] = get_vh("Heating Coil Heating Rate", "Main Heating Coil 1")
        self._var_handles['AHU_Fan_Power'] = get_vh("Fan Electricity Rate", "Supply Fan 1")

        # Zones
        target_zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]

        for z in target_zones:
            # States
            self._var_handles[f'{z}_Temp'] = get_vh("Zone Air Temperature", z)
            self._var_handles[f'{z}_Rad_Temp'] = get_vh("Zone Mean Radiant Temperature", z)
            self._var_handles[f'{z}_Humid'] = get_vh("Zone Mean Air Humidity Ratio", z)
            self._var_handles[f'{z}_CO2'] = get_vh("Zone Air CO2 Concentration", z)

            # Parameters (Loads & Occupancy)
            self._var_handles[f'{z}_Occupancy'] = get_vh("Zone People Occupant Count", z)
            # The equipment names usually follow a pattern in the IDF. E.g., "SPACE1-1 ELECEQ 1"
            self._var_handles[f'{z}_Equip_Power'] = get_vh("Zone Electric Equipment Total Heating Rate", f"{z} ELECEQ 1")

            # Terminal Actions
            self._var_handles[f'{z}_VAV_MassFlow'] = get_vh("System Node Mass Flow Rate", f"{z} In Node")
            self._var_handles[f'{z}_Reheat_Rate'] = get_vh("Heating Coil Heating Rate", f"{z} Zone Coil")

        self._fast_logger_ready = True
        print(f"[Logger] Successfully mapped {len(self._var_handles)} physical data points.")

    # --- 2. Runtime Data Extraction ---
    day = self.exchange.day_of_year(state)
    hour = self.exchange.current_time(state)

    row = {'Day': day, 'Hour': hour}

    for key, handle in self._var_handles.items():
        row[key] = self.exchange.get_variable_value(state, handle) if handle > 0 else np.nan

    self.sim_log_data.append(row)

# Bind and register
sim.simulation_logger = types.MethodType(simulation_logger, sim)
sim.register_handlers("after_zone", [{"method_name": "simulation_logger"}])

['simulation_logger']

In [14]:
# @title Run Simulation

sim.set_simulation_params(
    start=(1, 1),
    end=(1, 7),
    timestep_per_hour = 4, # 4 (every 15 minutes) or 6 (every 10 minutes).
    start_day_of_week="Monday",
)

print("Starting EnergyPlus Simulation...")
res = sim.run_annual()

if(res == 0):
    print("Simulation Complete!")

if(res == 1):
    err_path = Path(OUT_DIR) / "eplusout.err"
    if err_path.exists():
        print("--- EnergyPlus Error Log ---")
        with open(err_path, 'r') as f:
            print(f.read()[-4000:])
    else:
        print(f"Could not find the error file at: {err_path}")

EnergyPlus state has been reset.
Starting EnergyPlus Simulation...
Deleted output file: /simulation/eplus_out/eplusout.sql
Deleted output file: /simulation/eplus_out/eplusout.err
Deleted output file: /simulation/eplus_out/eplusout.audit
EnergyPlus state has been reset.
Simulation Complete!


In [16]:
# 2. Extract and format the data
if hasattr(sim, 'sim_log_data') and len(sim.sim_log_data) > 0:
    df_log = pd.DataFrame(sim.sim_log_data)

    # Create a continuous time axis for easy plotting
    df_log['Time_Hours'] = (df_log['Day'] - df_log['Day'].iloc[0]) * 24 + df_log['Hour']
    df_log.set_index('Time_Hours', inplace=True)

    # 3. Save to CSV
    csv_filename = "MPC_Simulation_Log.csv"
    df_log.to_csv(csv_filename)

    print(f"\n✅ Simulation Complete!")
    print(f"✅ Data Extracted! Shape: {df_log.shape}")
    print(f"✅ Data successfully saved to: {csv_filename}")

    # Display the first few rows
    display(df_log)
else:
    print("No log data found. The simulation may not have run correctly.")


✅ Simulation Complete!
✅ Data Extracted! Shape: (1152, 51)
✅ Data successfully saved to: MPC_Simulation_Log.csv


,Day,Hour,Env_Temp_Out,Env_Humid_Out,AHU_Mix_Temp,AHU_Cool_Out_Temp,AHU_Heat_Out_Temp,AHU_Supply_Temp,AHU_Cooling_Rate,AHU_Heating_Rate,...,SPACE4-1_VAV_MassFlow,SPACE4-1_Reheat_Rate,SPACE5-1_Temp,SPACE5-1_Rad_Temp,SPACE5-1_Humid,SPACE5-1_CO2,SPACE5-1_Occupancy,SPACE5-1_Equip_Power,SPACE5-1_VAV_MassFlow,SPACE5-1_Reheat_Rate
Time_Hours,,,,,,,,,,,,,,,,,,,,,
0.25,1,0.25,NaN,0.017278,25.284157,25.284157,25.284157,26.239277,0.0,0.0,...,0.091488,0.0,22.727218,21.956400,0.017503,591.819277,0.0,NaN,0.089169,0.0
0.50,1,0.50,NaN,0.017495,25.103839,25.103839,25.103839,26.059200,0.0,0.0,...,0.091488,0.0,22.676548,21.892473,0.017447,583.109040,2.0,NaN,0.089169,0.0
0.75,1,0.75,NaN,0.017697,24.930773,24.930773,24.930773,25.886363,0.0,0.0,...,0.091488,0.0,22.626131,21.838380,0.017393,581.116387,2.0,NaN,0.089169,0.0
1.00,1,1.00,NaN,0.017884,24.775642,24.775642,24.775642,25.731405,0.0,0.0,...,0.091488,0.0,22.562686,21.788667,0.017324,582.278940,2.0,NaN,0.089169,0.0
1.25,1,1.25,NaN,0.017829,24.636907,24.636907,24.636907,25.592806,0.0,0.0,...,0.091488,0.0,22.494925,21.739113,0.017251,584.684037,2.0,NaN,0.089169,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167.00,7,23.00,NaN,0.016804,25.573948,25.573948,25.573948,26.527922,0.0,0.0,...,0.091488,0.0,23.417247,22.261889,0.018271,1121.110395,4.0,NaN,0.089169,0.0
167.25,7,23.25,NaN,0.016819,25.430033,25.430033,25.430033,26.384212,0.0,0.0,...,0.091488,0.0,23.349064,22.212506,0.018193,1118.584645,4.0,NaN,0.089169,0.0
167.50,7,23.50,NaN,0.016830,25.291899,25.291899,25.291899,26.246267,0.0,0.0,...,0.091488,0.0,23.327160,22.170908,0.018169,1125.462010,6.0,NaN,0.089169,0.0
